### The Training Problem

Quantization solves the problem of loading a huge model in memory and for using it for inferencing but quantized n-bit weights **cannot be trained directly:**
- They're n-bit integers, not continuous floats
- Can't compute gradients for discrete integers
- If you dequantize to FP16, train, then re-quantize → massive information loss (apart from the fact that it won't fit in memory as was the bottleneck originally)

### 1. The Solution: QLoRA (Quantized LoRA)

Keep the n-bit weights **frozen**, train only **low-rank adapters**:

Frozen n-bit base weights
       ↓ (dequantized for forward pass)
   [Output of base layer]
       ↓
   LoRA adapters (trainable, FP16)
       ↓
   [Combined output with gradient flow]



**During training:**

1. Forward pass: dequantize n-bit weights layer-by-layer (same as inference)
2. Compute: `output = base_output + LoRA_A @ LoRA_B`
3. Backward: gradients only flow into `LoRA_A` and `LoRA_B`, not the n-bit weights
4. Update: only LoRA parameters get trained (tiny, like 1-2% of full model)

### What This Means

**BitsAndBytes alone:** ❌ Can't finetune  
**BitsAndBytes + LoRA:** ✅ Can finetune efficiently

The n-bit weights stay frozen in memory, you only compute gradients for the LoRA adapters.

### 2. How Does LoRA Work?

Two things:
1. Singular Vector Decomposition (roughly, we don't actually break down the weights into two)
2. Choosing a small number for rank where rank is (n, rank) * (rank, m) a.k.a the facing dimension.

Instead of using a big matrix as adapter (which would have to be the same size as the original layer that was quantized) to store the changes (the updates), we’ll replace it with two small matrices. 
Decomposing matrices inevitably leads to losses. The smaller the two resulting matrices are, the worse the approximation is. But we’re not doing that. Instead, we’re starting with two small matrices (let’s call them A and B) which will have their weights updated during training. If we multiply them, we’ll get a full-sized matrix. This resulting matrix would represent the accumulated updates to the, now quantized and frozen, weights. In reality, we don’t need to compute this matrix; we can forward our inputs through the two small matrices in sequence and achieve the same output.


Even though the matrix is full-sized, most of its values are just linear combinations of
other values. The rank shows you the real dimensions of the matrix—or basically, how much redundancy is in there:

- A high-ranking value (that is, one close to the actual number of rows or columns) means there’s little to no redundancy.
- A low-rank matrix implies there’s a lot of redundancy, and the large matrix can be easily represented by two smaller ones.


Now, you know why it’s called a low-rank adapter. We’re using two small matrices to produce a low-rank big matrix (representing the updates) that matches the size of the original layer.

</br>

NOTE:
While low-rank matrices may be sufficient for fine-tuning, they can **severely impede** the learning ability of a model being trained from scratch. The low-rank nature of the matrices acts as a constrain, **limiting the model’s capacity to explore and learn complex relationships, effectively functioning as if it were a much smaller model**.
A pre-trained model, however, has already learned these relationships. Fine-tuning, as the name suggests, introduces only small changes to the underlying model, which can be effectively represented by low-rank matrices

### 2.1 Working out a code example

This is our original full layer (as it would be in a model, among many many layers)

```python 
base_layer = nn.Linear(1024, 1024, bias=False)
```

Now we make smaller matrices for it that multiple upto become the same size.

```python
torch.manual_seed(11)
r = 8
layer_A = nn.Linear(base_layer.in_features, r, bias=False)
layer_B = nn.Linear(r, base_layer.out_features, bias=False)
```

Now the forward pass should look like this:
```python
torch.manual_seed(19)
batch = torch.randn(1, 1024)
batch @ (base_layer.weight.data + layer_B.weight @ layer_A.weight).T
```

With the distributive property of matrix multiplication, we can split this into two passes: one forward pass through the base layer, and another through the resulting low-rank matrix just like p . (q+r) = p.q + p.r

```python
regular_output = batch @ base_layer.weight.data.T
additional_output = batch @ (layer_B.weight @ layer_A.weight).T
regular_output, additional_output
```

Another addition property allows us to chain the operation one on the other for `additional_output`. The whole forward pass would look something like below:
```python
regular_output = base_layer(batch)
out_A = layer_A(batch)
additional_output = layer_B(out_A)
alpha = 2*r
output = regular_output + (alpha / r) * additional_output
```

And we have a forward pass with LoRA!!


The denominator is fixed and it’s the rank we chose, but we can tweak the numerator, which is called LoRA’s
alpha. In practice, alpha’s often set to **twice** the rank, effectively doubling the additional output.


### 2.2 What are components we training?

### 🔍 What’s left to be trained when attention heads are frozen?
- **Massive linear layers in attention heads** (the query, key, value, and output projections) are often too large to retrain directly — they consume huge amounts of memory and compute.  
- When these are frozen, fine-tuning typically focuses on **smaller, more efficient components**, such as:
  - **LayerNorm parameters** (scale and bias terms that stabilize training).
  - **Bias terms in feed-forward layers** (much smaller than full weight matrices).
  - **Adapter layers** (lightweight trainable modules inserted between frozen layers).
  - **LoRA (Low-Rank Adaptation) matrices** — small rank-decomposition updates applied to frozen weight matrices.
  - **Embedding layers** (sometimes partially trainable, depending on the setup).

So, instead of retraining the “brain” of the Transformer (attention heads), fine-tuning works on the “knobs and dials” that adjust how the frozen parts behave.

---

### 💾 What data type do these trainable parameters have?
- The frozen attention weights are **quantized** (e.g., int8 or int4) to save space.  
- The remaining trainable parameters (like adapters, biases, or LoRA matrices) are usually kept in **higher precision formats**:
  - **FP16 (half precision)** or **bfloat16** for efficiency on modern GPUs/TPUs.
  - Sometimes **FP32** if stability is critical, though this is less common in large-scale fine-tuning because of memory cost.


---

In [16]:
from collections import Counter

def get_parm_dtypes(iterable, top_k=3):
    return Counter([p.dtype for p in iterable]).most_common(top_k)

In [17]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

print("=== LOADING STANDARD HF 4-BIT MODEL ===")

torch.cuda.empty_cache()

model_name = "facebook/opt-350m"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4 — the actual 4-bit data type
    bnb_4bit_use_double_quant=True,     # Double quantization (extra memory saving)
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_q4 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map='cuda:0',
    dtype=torch.bfloat16,  # Mixed precision (bf16 or fp16) can help reduce memory usage and speed up training
    low_cpu_mem_usage=True,      # Helps reduce peak during load
    trust_remote_code=True
)

=== LOADING STANDARD HF 4-BIT MODEL ===


Loading weights: 100%|██████████| 388/388 [00:00<00:00, 736.99it/s]


In [18]:
print(model_q4.get_memory_footprint()/1e6, get_parm_dtypes(model_q4.parameters()))

207.835136 [(torch.bfloat16, 242), (torch.uint8, 146)]


In [19]:
def trainable_parms(model):
    parms = [(name, param.dtype) for name, param in model.named_parameters() if param.requires_grad]
    return parms
tp = trainable_parms(model_q4.model)
len(tp), tp

(242,
 [('decoder.embed_tokens.weight', torch.bfloat16),
  ('decoder.embed_positions.weight', torch.bfloat16),
  ('decoder.layers.0.self_attn.k_proj.bias', torch.bfloat16),
  ('decoder.layers.0.self_attn.v_proj.bias', torch.bfloat16),
  ('decoder.layers.0.self_attn.q_proj.bias', torch.bfloat16),
  ('decoder.layers.0.self_attn.out_proj.bias', torch.bfloat16),
  ('decoder.layers.0.self_attn_layer_norm.weight', torch.bfloat16),
  ('decoder.layers.0.self_attn_layer_norm.bias', torch.bfloat16),
  ('decoder.layers.0.fc1.bias', torch.bfloat16),
  ('decoder.layers.0.fc2.bias', torch.bfloat16),
  ('decoder.layers.0.final_layer_norm.weight', torch.bfloat16),
  ('decoder.layers.0.final_layer_norm.bias', torch.bfloat16),
  ('decoder.layers.1.self_attn.k_proj.bias', torch.bfloat16),
  ('decoder.layers.1.self_attn.v_proj.bias', torch.bfloat16),
  ('decoder.layers.1.self_attn.q_proj.bias', torch.bfloat16),
  ('decoder.layers.1.self_attn.out_proj.bias', torch.bfloat16),
  ('decoder.layers.1.self_attn_

BitsAndBytes automatically sets requires_grad=False for quantized layers.

The quantized weights are frozen by default—they can't be trained. This makes sense because:

They're stored as 4-bit/8-bit integers, not continuous floats
Can't compute meaningful gradients for discrete values
To Enable Training with LoRA
You need to explicitly prepare the model which we will see below.

---

### 3. What does `prepare_model_for_kbit_training()` do?

It prepares a **quantized model** (typically loaded in 4-bit or 8-bit using `bitsandbytes`) for training with adapters like **LoRA** or **QLoRA**.

### What it actually does (main steps):

1. **Freezes the base model parameters**  
   It sets `requires_grad = False` for ALL parameters in the original (quantized) model.

2. **Upcasts certain layers to float32 (fp32) for numerical stability**  
   Quantized models (especially 4-bit NF4) often use low precision (fp16/bf16) for computations, which can lead to instabilities like NaN losses. The function casts:
   - All **normalization layers** (LayerNorm, RMSNorm, etc.) to fp32 — these involve reduction operations (sum/mean) that are sensitive to low precision.
   - **Input embeddings** (`model.get_input_embeddings().weight`)
   - **Output embeddings / LM head** (`model.get_output_embeddings().weight` or `lm_head`)

   But often due to memory limitations, bf16 is chosen as the higher precision dtype instead of fp32.

3. **Enables gradient checkpointing** (by default)  
   It calls `model.gradient_checkpointing_enable()` (unless you pass `use_gradient_checkpointing=False`). This trades compute for memory, which is essential when fine-tuning large models on limited GPU VRAM. In more recent versions, this can be achieved more easily using a keyword argument for gradient `checkpointing: {'use_reentrant': False}`.


### Key notes:

- Disabling gradient checkpointing is possible via `prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)` if you have enough memory or want faster training.

This function is a critical part of making **QLoRA** work efficiently and stably on consumer hardware. Without it, you often encounter unstable training (NaNs) or higher memory usage.

In [20]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

prepared_model = prepare_model_for_kbit_training(model_q4, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False}, )
prepared_model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear4bit(in_features=1024, out_features=512, bias=False)
      (project_in): Linear4bit(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear4bit(in_features=1024, out_features=4096, bias=True)
          (

In [21]:
trainable_parms(prepared_model)


[]

In [22]:
def parms_of_dtype(model, dtype=torch.float32):
    parms = [name for name, param in model.named_parameters() if param.dtype == dtype]
    return parms
parms_of_dtype(prepared_model),  parms_of_dtype(prepared_model, dtype=torch.bfloat16)

(['model.decoder.embed_tokens.weight',
  'model.decoder.embed_positions.weight',
  'model.decoder.layers.0.self_attn.k_proj.bias',
  'model.decoder.layers.0.self_attn.v_proj.bias',
  'model.decoder.layers.0.self_attn.q_proj.bias',
  'model.decoder.layers.0.self_attn.out_proj.bias',
  'model.decoder.layers.0.self_attn_layer_norm.weight',
  'model.decoder.layers.0.self_attn_layer_norm.bias',
  'model.decoder.layers.0.fc1.bias',
  'model.decoder.layers.0.fc2.bias',
  'model.decoder.layers.0.final_layer_norm.weight',
  'model.decoder.layers.0.final_layer_norm.bias',
  'model.decoder.layers.1.self_attn.k_proj.bias',
  'model.decoder.layers.1.self_attn.v_proj.bias',
  'model.decoder.layers.1.self_attn.q_proj.bias',
  'model.decoder.layers.1.self_attn.out_proj.bias',
  'model.decoder.layers.1.self_attn_layer_norm.weight',
  'model.decoder.layers.1.self_attn_layer_norm.bias',
  'model.decoder.layers.1.fc1.bias',
  'model.decoder.layers.1.fc2.bias',
  'model.decoder.layers.1.final_layer_norm.we

In [23]:
prepared_model.get_memory_footprint()/1e6 # increase due to moving some parameters to fp32 for training

264.15104

### 4. Time for LoRA - Adding the adapters

We use PEFT (Parameter Efficient Fine-Tuning) package to implement LoRA. PEFT is integrated with the Transformers, Diffusers, and Accelerate libraries to provide a faster and easier way to
load, train, and use large models for inference.


As you can see, there are several options available for you to configure, starting with the adapters themselves,
which include:
- r: the rank of the LoRA adapters. Typical values are 8, 16, and 32.
- lora_alpha: the scaling parameter. The typical value is 2*r so, effectively speaking, the adapter’s output is multiplied by two.
- lora_dropout: the dropout probability of LoRA layers. Dropout for LoRA layers is usually set to a very low rate, e.g. 0.05.
- bias: none, all, or lora_only, it determines if the biases are being updated during training 
- modules_to_save: When you fine-tune a model with LoRA, the original model weights are frozen and only the LoRA adapter weights are trained. However, sometimes you also want to train and save certain original layers (e.g., classification heads, embedding layers) in addition to LoRA weights modules_to_save lets you specify these layers by name so that:

    - They are not frozen during training (requires_grad=True).
    - They are saved when you call model.save_pretrained()
    - If you're only doing standard instruction tuning without new tokens → you usually don't need modules_to_save.
    - If you're adding new tokens to the vocabulary (chat templates, function calling, etc.) → add "embed_tokens" (and often "lm_head") to modules_to_save.
    - This tells PEFT: "Train these layers fully and save their updated weights along with the LoRA adapters."


While updating the biases may contribute to the model’s performance, it "means that, even when disabling the adapters, the model will not produce the same output as the base model would have without adaptation", so, for the sake of having more flexibility down the line, we’re keeping the bias unchanged. 

The task_type argument, since we’re handling LLMs, is always going to be "CAUSAL_LM".

In [24]:
config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['embed_tokens', 'q_proj', 'v_proj'],  # For OPT models, these are the attention projection layers. You can also specify a list of layer names or regex patterns to match layer names.
    modules_to_save=['layer_norm'],   # moved embedding layer to target_modules because training it directly is very expensive so we rather attach a LoRA adapter to it and train that adapter instead
)

In [25]:
config.target_modules

{'embed_tokens', 'q_proj', 'v_proj'}

- How does it determine which layers should have adapters? 
: Popular architectures come with preconfigured lists of target layers where the adapters will be applied.

- Are only quantized layers eligible? 
: The target layers will receive adapters whether they’re quantized or not; although quantized layers are
natural candidates because they cannot be updated.

- Can I attach adapters to regular (non-quantized) linear layers as well?
: Yes, you can, any Linear (or Conv1D!) layer can have an adapter to call its own.

BUT
- Since there are new models and architectures being released on a weekly basis, chances are that there is no preconfigured list of target layers in your currently installed version of the PEFT library.
- If that happens, you’ll need to look up the names of the linear layers in the attention module on your own and pass it to `target_modules`

In [26]:
from peft.utils.constants import TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING
TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING.keys()

dict_keys(['t5', 'mt5', 'bart', 'gpt2', 'bloom', 'blip-2', 'opt', 'gptj', 'gpt_neox', 'gpt_neo', 'bert', 'roberta', 'xlm-roberta', 'electra', 'deberta-v2', 'deberta', 'layoutlm', 'llama', 'llama4', 'chatglm', 'gpt_bigcode', 'mpt', 'RefinedWebModel', 'RefinedWeb', 'falcon', 'btlm', 'codegen', 'mistral', 'mixtral', 'stablelm', 'phi', 'gemma', 'gemma2', 'gemma3_text', 'qwen2', 'qwen3', 'rwkv', 'rwkv7'])

In [27]:
# no mapping for phi-3 but we can see the same for phi
TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING['phi']

['q_proj', 'v_proj', 'fc1', 'fc2']

### 4.1 Difference between **`target_modules`** and **`modules_to_save`** in `LoraConfig`
### 1. `target_modules` — Where to apply **LoRA adapters**

This is the **main** parameter for LoRA.

- It tells PEFT **which layers** of the base model should get **LoRA** (low-rank adaptation) applied.
- For each module listed here, PEFT **replaces** the original weight matrix with two small low-rank matrices (`lora_A` and `lora_B`).
- Only these small LoRA matrices are trained (very few parameters).
- The original weights stay **frozen**.

**Typical values:**
```python
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]          # common for Llama
# or
target_modules = "all-linear"     # modern shortcut (applies to all linear layers)
```

**Goal**: Efficient fine-tuning by updating only a tiny fraction of parameters via adapters.

### 2. `modules_to_save` — Modules to **fully train and save** (no LoRA)

This is a completely different concept.

- Modules listed here are **not** adapted with LoRA.
- Instead, they are made **fully trainable** (all their parameters get `requires_grad = True`).
- PEFT creates a **copy** of these modules and trains the copy.
- These full weights are saved in the adapter checkpoint (alongside the LoRA weights).

**Most common use case:**
- When you **add new tokens** to the tokenizer → you resize the embedding layer (`embed_tokens`) and the language modeling head (`lm_head`).
- The new rows are randomly initialized, so you need to **fully fine-tune** them.


### Side-by-side comparison

| Aspect                      | `target_modules`                              | `modules_to_save`                                      |
|----------------------------|-----------------------------------------------|--------------------------------------------------------|
| What happens to the layer  | LoRA adapters (A+B matrices) are added        | No LoRA — the entire layer is fully trainable          |
| Training method            | Only small LoRA weights are updated           | All original parameters are updated                    |
| Parameter count impact     | Very small increase (e.g. +0.1–1%)            | Large increase (full size of the layer)                |
| Memory & speed             | Very efficient                                | More expensive (especially for embeddings + lm_head)   |
| When to use                | Attention / MLP layers (q_proj, gate_proj…)   | Embeddings & head when adding new tokens               |
| Saved in checkpoint        | Only the small LoRA weights                   | Full updated weights of these modules                  |
| Default                    | You usually specify it                        | Usually `None` (unless adding tokens)                  |

### Practical example (most common in QLoRA)

```python
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],  # ← LoRA goes here
    modules_to_save=["embed_tokens", "lm_head"],   # ← Full fine-tuning here
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
```

### Quick rule of thumb

- Use **`target_modules`** → for **efficient** adaptation of the main transformer blocks.
- Use **`modules_to_save`** → when you need to **fully update** specific layers (almost always `embed_tokens` and `lm_head` after resizing the tokenizer).


### 4.2 Attaching the LoRA adapter weights

The configuration alone may be enough if you’re training with the SFTTrainer class, which takes care of attaching the adapters to the base model under the hood.

In [28]:
peft_model = get_peft_model(prepared_model, config, adapter_name='default')
peft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): lora.Embedding(
            (base_layer): Embedding(50272, 512, padding_idx=1)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict()
            (lora_B): ModuleDict()
            (lora_embedding_A): ParameterDict(  (default): Parameter containing: [torch.cuda.FloatTensor of size 8x50272 (cuda:0)])
            (lora_embedding_B): ParameterDict(  (default): Parameter containing: [torch.cuda.FloatTensor of size 512x8 (cuda:0)])
            (lora_magnitude_vector): ModuleDict()
          )
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear4bit(in_features=1024, out_features=512, bias=False)
          (project_in): Linear4bit(in_features=512, out_features=1024, bias=False)
          (layers):

In [29]:
lin = (peft_model.base_model.model.model.decoder.layers[0].self_attn.q_proj)
lin

lora.Linear4bit(
  (base_layer): Linear4bit(in_features=1024, out_features=1024, bias=True)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.05, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=1024, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=1024, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)

In [30]:
peft_model.print_trainable_parameters()

trainable params: 1,291,008 || all params: 332,487,424 || trainable%: 0.3883


In [31]:
# to check the configuration of the peft model (how many adapters)
peft_model.peft_config.keys()

dict_keys(['default'])

Our model configuration is fit for fine-tuning. Now we have to focus on the data being input and how the tokenizer which is more or less the first point of contact of the input with the model parses and handles it.